In [1]:
!pip install lightkurve astropy wotan

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.1/261.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 80.7 MB/s eta 0:00:00
  Created wheel for fbpca: filename=fbpca-1.0-py3-none-any.whl size=11373 sha256=46f968c6f2a97800e2feda850b8fc3a073db75b5040bceeff8607177b6bf7f4d
  Stored in directory: /root/.cache/pip/wheels/04/15/cd/2f622795b09e83471a3be5d2581cd9cf96a6ec7aa78e8deffe
  Created wheel for memoization:

In [2]:
import zipfile

zip_path = "/content/TESS_LightCurves.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/TESS_Data")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [3]:
import os

tess_files = []

# Search only inside TESS folder
for root, dirs, files in os.walk("/content/TESS_Data/mastDownload/TESS"):
    for file in files:
        if file.endswith(".fits") or file.endswith(".fits.gz"):
            tess_files.append(os.path.join(root, file))

print("Total TESS FITS files found:", len(tess_files))

for file in tess_files:
    print(file)

Total TESS FITS files found: 10
/content/TESS_Data/mastDownload/TESS/tess2019279210107-s0017-0000000083952938-0161-s/tess2019279210107-s0017-0000000083952938-0161-s_lc.fits
/content/TESS_Data/mastDownload/TESS/tess2022273165103-s0057-0000000083940523-0245-s/tess2022273165103-s0057-0000000083940523-0245-s_lc.fits
/content/TESS_Data/mastDownload/TESS/tess2019279210107-s0017-0000000083952046-0161-s/tess2019279210107-s0017-0000000083952046-0161-s_lc.fits
/content/TESS_Data/mastDownload/TESS/tess2019279210107-s0017-0000000083940523-0161-s/tess2019279210107-s0017-0000000083940523-0161-s_lc.fits
/content/TESS_Data/mastDownload/TESS/tess2024274222008-s0084-0000000083940523-0281-a_fast/tess2024274222008-s0084-0000000083940523-0281-a_fast-lc.fits
/content/TESS_Data/mastDownload/TESS/tess2019279210107-s0017-0000000083953033-0161-s/tess2019279210107-s0017-0000000083953033-0161-s_lc.fits
/content/TESS_Data/mastDownload/TESS/tess2024274222008-s0084-0000000083940523-0281-s/tess2024274222008-s0084-000

In [4]:
os.makedirs("/content/Clean_CSV", exist_ok=True)
os.makedirs("/content/Plots", exist_ok=True)

print("Output folders created successfully!")

Output folders created successfully!


In [5]:
from astropy.io import fits
from astropy.stats import sigma_clip
from wotan import flatten

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

for i, file in enumerate(tess_files):

    print(f"\nProcessing file {i+1}/{len(tess_files)}")

    try:

        # -------------------------
        # Open FITS file
        # -------------------------

        hdul = fits.open(file)
        data = hdul[1].data

        # -------------------------
        # Extract columns
        # -------------------------

        time = data['TIME']
        quality = data['QUALITY']

        if 'PDCSAP_FLUX' in data.columns.names:
            flux = data['PDCSAP_FLUX']
        else:
            flux = data['SAP_FLUX']

        # Save raw copy
        raw_time = time.copy()
        raw_flux = flux.copy()

        # -------------------------
        # Remove NaN values
        # -------------------------

        mask = ~np.isnan(time) & ~np.isnan(flux)

        time = time[mask]
        flux = flux[mask]
        quality = quality[mask]

        # -------------------------
        # Keep only QUALITY = 0
        # -------------------------

        good = quality == 0

        time = time[good]
        flux = flux[good]

        # -------------------------
        # Normalize flux
        # -------------------------

        flux = flux / np.median(flux)

        # -------------------------
        # Sigma clipping
        # -------------------------

        clipped_flux = sigma_clip(flux, sigma=5)

        mask = ~clipped_flux.mask

        clean_time = time[mask]
        clean_flux = clipped_flux.data[mask]

        # -------------------------
        # Detrending
        # -------------------------

        flatten_flux = flatten(clean_time, clean_flux)

        # -------------------------
        # Save cleaned CSV
        # -------------------------

        df = pd.DataFrame({
            'TIME': clean_time,
            'FLUX': flatten_flux
        })

        csv_name = os.path.basename(file)
        csv_name = csv_name.replace(".fits", ".csv")
        csv_name = csv_name.replace(".gz", "")

        df.to_csv(f"/content/Clean_CSV/{csv_name}",
                  index=False)

        # -------------------------
        # Plot before and after
        # -------------------------

        plt.figure(figsize=(12,5))

        plt.subplot(1,2,1)
        plt.plot(raw_time, raw_flux, '.', markersize=1)
        plt.title("Before Cleaning")
        plt.xlabel("Time")
        plt.ylabel("Flux")

        plt.subplot(1,2,2)
        plt.plot(clean_time,
                 flatten_flux,
                 '.',
                 markersize=1)

        plt.title("After Cleaning")
        plt.xlabel("Time")
        plt.ylabel("Normalized Flux")

        plt.tight_layout()

        plot_name = os.path.basename(file)
        plot_name = plot_name.replace(".fits", ".png")
        plot_name = plot_name.replace(".gz", "")

        plt.savefig(f"/content/Plots/{plot_name}",
                    dpi=300)

        plt.close()

        print("Successfully processed!")

    except Exception as e:
        print("Error processing file:", e)


Processing file 1/10
Successfully processed!

Processing file 2/10
Successfully processed!

Processing file 3/10
Successfully processed!

Processing file 4/10
Successfully processed!

Processing file 5/10
Successfully processed!

Processing file 6/10
Successfully processed!

Processing file 7/10
Successfully processed!

Processing file 8/10
Successfully processed!

Processing file 9/10
Successfully processed!

Processing file 10/10
Successfully processed!


In [6]:
print(os.listdir("/content/Clean_CSV"))

['tess2019279210107-s0017-0000000083952938-0161-s_lc.csv', 'tess2019279210107-s0017-0000000083952046-0161-s_lc.csv', 'tess2019279210107-s0017-0000000083940523-0161-s_lc.csv', 'tess2019279210107-s0017-0000000083953033-0161-s_lc.csv', 'tess2024274222008-s0084-0000000083939881-0281-s_lc.csv', 'tess2024274222008-s0084-0000000083940523-0281-a_fast-lc.csv', 'tess2024274222008-s0084-0000000083940523-0281-s_lc.csv', 'tess2022273165103-s0057-0000000083940523-0245-s_lc.csv', 'tess2022273165103-s0057-0000000083940523-0245-a_fast-lc.csv', 'tess2022273165103-s0057-0000000083939881-0245-s_lc.csv']


In [7]:
print(os.listdir("/content/Plots"))

['tess2024274222008-s0084-0000000083940523-0281-a_fast-lc.png', 'tess2022273165103-s0057-0000000083940523-0245-s_lc.png', 'tess2022273165103-s0057-0000000083939881-0245-s_lc.png', 'tess2019279210107-s0017-0000000083952046-0161-s_lc.png', 'tess2019279210107-s0017-0000000083952938-0161-s_lc.png', 'tess2019279210107-s0017-0000000083953033-0161-s_lc.png', 'tess2024274222008-s0084-0000000083940523-0281-s_lc.png', 'tess2024274222008-s0084-0000000083939881-0281-s_lc.png', 'tess2019279210107-s0017-0000000083940523-0161-s_lc.png', 'tess2022273165103-s0057-0000000083940523-0245-a_fast-lc.png']
